# Traductor Inglés → Español con un Transformer (Modelo Local)

En este cuaderno implementaremos un traductor de inglés a español usando un modelo **Transformer** de forma local, descargando el modelo directamente desde Hugging Face.

### ¿Qué modelo usamos?
- **`Helsinki-NLP/opus-mt-en-es`**, de la familia **MarianMT**: un Transformer encoder-decoder (secuencia a secuencia) entrenado específicamente para traducción inglés → español.
- Es pequeño (~75 millones de parámetros), por lo que corre sin problema en la CPU o en la GPU gratuita de Colab.
- Es el mismo modelo que usamos por API en el cuaderno **10-04-Traductor-HF-API**.

### Ventaja del modelo local
- **No necesita API Key** ni conexión a internet después de la descarga inicial.
- Funciona completamente offline una vez descargado.
- No tiene costos de uso ni límites de llamadas.

> Para comparar con un modelo de lenguaje actual de propósito general, mira el cuaderno **10-05-Traductor-GPT6**, donde hacemos la misma tarea con GPT-6.

## Paso 1: Instalación de las bibliotecas necesarias

Instalamos `transformers` de Hugging Face y `sentencepiece` (tokenizador que usan los modelos MarianMT).

In [1]:
# Instalamos transformers (modelos preentrenados) y sentencepiece (tokenizador de MarianMT)
!pip install -q transformers sentencepiece

## Paso 2: Cargar el modelo preentrenado

Usamos `AutoTokenizer` y `AutoModelForSeq2SeqLM`, que detectan automáticamente
la arquitectura del modelo.

El modelo `Helsinki-NLP/opus-mt-en-es` es un traductor inglés → español de tipo MarianMT.

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Modelo MarianMT entrenado para traducción inglés -> español
model_name = "Helsinki-NLP/opus-mt-en-es"

# Descargamos el tokenizador (convierte texto en tokens numéricos)
# y el modelo preentrenado para traducción
print("Descargando tokenizador y modelo...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
print("Modelo cargado exitosamente!")

Descargando tokenizador y modelo...


config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.59M [00:00<?, ?B/s]

... (4 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  312MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Modelo cargado exitosamente!


## Paso 3: Definir la función de traducción

Esta función toma un texto en inglés, lo tokeniza, lo pasa por el modelo,
y decodifica la salida para obtener la traducción en español.

### Parámetros del modelo:
- `max_new_tokens`: longitud máxima de la traducción generada.
- `num_beams`: cantidad de "caminos" que explora el modelo para encontrar la mejor traducción (beam search).
- `early_stopping`: detiene la generación cuando todas las hipótesis del beam search terminan.

> A diferencia de T5, este modelo **no necesita un prefijo** como "translate English to Spanish:": ya está entrenado solo para esta tarea.

In [3]:
def translate_english_to_spanish(text):
    """
    Traduce texto de inglés a español usando el modelo MarianMT local.

    Args:
        text (str): Texto en inglés a traducir.

    Returns:
        str: Texto traducido al español.
    """
    # Tokenizamos el texto (lo convertimos a números que el modelo entiende)
    # return_tensors="pt" indica que queremos tensores de PyTorch
    inputs = tokenizer(text, return_tensors="pt", truncation=True)

    # Generamos la traducción usando beam search
    # num_beams=4 explora 4 posibles traducciones y elige la mejor
    output_ids = model.generate(
        **inputs,
        max_new_tokens=200,
        num_beams=4,
        early_stopping=True
    )

    # Decodificamos la salida (de números a texto legible)
    # skip_special_tokens=True omite tokens como <pad> o </s>
    translated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return translated_text

## Paso 4: Ejemplo práctico

Probemos el traductor con un párrafo sobre inteligencia artificial.

In [4]:
# Texto de ejemplo en inglés
english_paragraph = (
    "Artificial intelligence is transforming industries around the world. "
    "It has applications in healthcare, education, and many other fields. "
    "The potential of AI to improve lives is enormous, but it also raises ethical concerns."
)

# Llamamos a la función para traducir
spanish_translation = translate_english_to_spanish(english_paragraph)

# Mostramos los resultados
print("Texto original en inglés:")
print(english_paragraph)
print("\nTraducción al español:")
print(spanish_translation)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Texto original en inglés:
Artificial intelligence is transforming industries around the world. It has applications in healthcare, education, and many other fields. The potential of AI to improve lives is enormous, but it also raises ethical concerns.

Traducción al español:
La inteligencia artificial está transformando industrias en todo el mundo. Tiene aplicaciones en salud, educación y muchos otros campos. El potencial de la IA para mejorar vidas es enorme, pero también plantea preocupaciones éticas.


## Paso 5: Más ejemplos

Probemos con oraciones más cortas para ver cómo se comporta el modelo.

In [5]:
# Lista de frases para traducir
frases = [
    "Machine learning is a subset of artificial intelligence.",
    "The robot can understand natural language and respond accordingly.",
    "Deep learning models require large amounts of data to train effectively."
]

for i, frase in enumerate(frases, 1):
    traduccion = translate_english_to_spanish(frase)
    print(f"Frase {i}:")
    print(f"  EN: {frase}")
    print(f"  ES: {traduccion}")
    print()

[transformers] Both `max_new_tokens` (=200) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Frase 1:
  EN: Machine learning is a subset of artificial intelligence.
  ES: El aprendizaje automático es un subconjunto de inteligencia artificial.



[transformers] Both `max_new_tokens` (=200) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Frase 2:
  EN: The robot can understand natural language and respond accordingly.
  ES: El robot puede entender el lenguaje natural y responder en consecuencia.

Frase 3:
  EN: Deep learning models require large amounts of data to train effectively.
  ES: Los modelos de aprendizaje profundo requieren grandes cantidades de datos para entrenar eficazmente.



## Para seguir explorando

- Este modelo es pequeño y rápido, pero un LLM actual suele traducir con más naturalidad y mejor manejo del contexto. Compáralo con el cuaderno **10-05-Traductor-GPT6**.
- Si tienes GPU, prueba **TranslateGemma** (Google, enero 2026; modelos abiertos de 4B, 12B y 27B parámetros para 55 idiomas), disponible en Hugging Face (requiere aceptar la licencia de Gemma).